In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Qwen2Model, Qwen2Config
from torch.distributions import Categorical
from transformers.cache_utils import DynamicCache

class VisualEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.in_channels = config["in_channels"]
        self.latent_dim = config["latent_dim"]
        self.input_hw = (config["grid_shape_x"], config["grid_shape_y"])
        
        layers = []
        prev_c = self.in_channels
        for i, out_c in enumerate(config["channels"]):
            layers.append(nn.Conv2d(
                prev_c, out_c, 
                kernel_size=config["kernel_size"], 
                padding=config["padding"], 
                stride=config["stride"] if i == len(config["channels"]) - 1 else 1
            ))
            layers.append(nn.ReLU())
            prev_c = out_c
            
        self.cnn = nn.Sequential(*layers)
        self._cnn_out_dim = self._infer_cnn_out_dim()
        self.fc = nn.Linear(self._cnn_out_dim, self.latent_dim)

    def _infer_cnn_out_dim(self):
        with torch.no_grad():
            dummy = torch.zeros(1, self.in_channels, *self.input_hw)
            out = self.cnn(dummy)
            return out.numel()

    def forward(self, x):
        # Handle both (B, C, H, W) and (B, T, C, H, W)  
        if x.dim() == 5:
            B, T, C, H, W = x.shape
            x = x.view(B * T, C, H, W)
            merge_time = True
        else:
            merge_time = False

        x = self.cnn(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)

        if merge_time:
            x = x.view(B, T, -1)
        return x

c:\Users\LeGat\Documents\GitHub\curry\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ActionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.model_name = config["model_name"]
        self.hidden_size = config["hidden_size"]
        self.vocab_size = config["vocab_size"]
        self.num_think_steps = config["num_think_steps"]
        
        if self.model_name == "qwen2":
            from transformers import Qwen2Model, Qwen2Config
            qwen_config = Qwen2Config(
                hidden_size=self.hidden_size,
                num_hidden_layers=config["args"]["num_layers"],
                num_attention_heads=config["args"]["num_attention_heads"],
                num_key_value_heads=config["args"]["num_attention_heads"],
                intermediate_size=self.hidden_size * 4,
                vocab_size=self.vocab_size
            )
            self.backbone = Qwen2Model(qwen_config)
        
        # PPO Heads
        self.actor_head = nn.Linear(self.hidden_size, self.vocab_size)
        self.value_head = nn.Linear(self.hidden_size, 1)

    def forward_step(self, x_t, past_key_values=None):
        h = x_t
        current_kv = past_key_values

        # Thinking Loop (Iterative Refinement)
        outputs = self.backbone(
            inputs_embeds=h,
            past_key_values=current_kv,
            use_cache=True
        )
        
        h_out = outputs.last_hidden_state # (B, 1, D)
        new_kv = outputs.past_key_values  # Tuple of KVs
        
        # Heads
        logits = self.actor_head(h_out)
        value = self.value_head(h_out)
        
        return {
            "logits": logits,      # (B, 1, Vocab)
            "value": value,        # (B, 1, 1)
            "hidden_state": h_out, # (B, 1, D)
            "past_key_values": new_kv
        }

    def forward_parallel(self, latents):
        """
        Full sequence parallel pass for Target Generation.
        latents: (B, T, D)
        """
        # Standard Causal Forward Pass
        outputs = self.backbone(
            inputs_embeds=latents,
            output_hidden_states=True,
            return_dict=True
        )
        
        h_seq = outputs.last_hidden_state # (B, T, D)
        
        logits = self.actor_head(h_seq)
        values = self.value_head(h_seq)

        return {
            "logits": logits,
            "values": values,
            "hidden_states": h_seq
        }

In [16]:
config_action_decoder = {
    "model_name": "qwen2",
    "hidden_size": 32,
    "vocab_size": 10,
    "num_think_steps": 1,
    "args": {
        "num_layers": 2,         
        "num_attention_heads": 1, 
    }
}

decoder = ActionDecoder(config_action_decoder)

x_t = torch.randn(1, 1, 32)
past_kv = None # First step has no history


# --- Test in your notebook ---
print("--- Testing Forward Step ---")
with torch.no_grad():
    out = decoder.forward_step(x_t, past_key_values=None)
    current_cache = out["past_key_values"]
for i in current_cache.to_legacy_cache():
    for j in i:
        print(j.requires_grad)



--- Testing Forward Step ---
False
False
False
False
